# Machine Learning for Texts: Detecting Toxic Comments

## Project Description

An online service launches a feature for communicating with customers on its website. A model is needed to detect negative messages so that they can be handled by the support team.

### Goals
- Train a model that classifies comments as positive or negative. A dataset with toxicity labels is available.
- Build a model with the quality metric *F1* of at least 0.75.

### Instructions
1. Load and prepare the data.
2. Train different models.
3. Draw conclusions.

### Data Description
The data is stored in the file `toxic_comments.csv`.
- **text** - text of the comment,
- **toxic** - target feature.

**Plan:**
- 1. Data Preparation
- 2. Model Training
    - 2.1 Logistic Regression
    - 2.2 NB-SVM
    - 2.3 Linear SVC
- 3. Conclusions

# 1. Data Preparation

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords as nltk_stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Load the data from the file and print the first 10 rows for a first look at the data.

In [2]:
comments = pd.read_csv('toxic_comments.csv')
comments.head(10)

,Unnamed: 0,text,toxic
0,0,Explanation\nWhy the edits made under my usern...,0
1,1,D'aww! He matches this background colour I'm s...,0
2,2,"Hey man, I'm really not trying to edit war. It...",0
3,3,"""\nMore\nI can't make any real suggestions on ...",0
4,4,"You, sir, are my hero. Any chance you remember...",0
5,5,"""\n\nCongratulations from me as well, use the ...",0
6,6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1
7,7,Your vandalism to the Matt Shirvington article...,0
8,8,Sorry if the word 'nonsense' was offensive to ...,0
9,9,alignment on this subject and which are contra...,0


In [3]:
comments.shape

(159292, 3)

In [4]:
comments['toxic'].value_counts()

0    143106
1     16186
Name: toxic, dtype: int64

In [5]:
print(f"Share of class 1 objects in the whole dataset: {(sum(comments['toxic']) / len(comments) * 100):.2f}%")

Share of class 1 objects in the whole dataset: 10.16%


Prepare the data for vectorization.
- Convert the character encoding to Unicode
- Lemmatize the words with `WordNetLemmatizer()` from `nltk`
- Remove punctuation and extra spaces
- Remove stop words (the list is loaded here, the words are removed during the tf-idf vectorization)

In [6]:
lemmatizer = WordNetLemmatizer()
def lemmatize(text):
    word_list = nltk.word_tokenize(text)
    lemmatized_output = ' '.join([lemmatizer.lemmatize(w) for w in word_list])
    return lemmatized_output

In [7]:
def clear_text(text):
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    return " ".join(text.split())

In [8]:
corpus = comments['text'].values.astype('U')

In [9]:
corpus_lemm = [lemmatize(clear_text(corpus[i])) for i in range(len(corpus))]

In [10]:
stopwords = set(nltk_stopwords.words('english'))

- Split the data into training and test sets

In [11]:
X_train, X_test, y_train, y_test = train_test_split(corpus_lemm, comments['toxic'], 
                                                    test_size=0.2,
                                                    random_state=42)

In [12]:
print(f"Training corpus size: {len(X_train)}")
print(f"Test corpus size: {len(X_test)}")

Training corpus size: 127433
Test corpus size: 31859


- Vectorize the corpora with `TfidfVectorizer` and remove the stop words at the same time.
- First try to train the model without n-grams.

In [13]:
tf_idf_vec = TfidfVectorizer(ngram_range=(1,1), stop_words=list(stopwords),
               min_df=3, max_df=0.9, strip_accents='unicode', use_idf=True,
               smooth_idf=True, sublinear_tf=True)

In [14]:
X_train_vec = tf_idf_vec.fit_transform(X_train)

In [15]:
X_test_vec = tf_idf_vec.transform(X_test)

In [16]:
print(f"Training dataset shape: {X_train_vec.shape}")
print(f"Test dataset shape: {X_test_vec.shape}")

Training dataset shape: (127433, 41662)
Test dataset shape: (31859, 41662)


# 2. Model Training

Find the accuracy of a constant model that predicts all comments as non-toxic ('toxic' = 0).

In [17]:
base_predicts = pd.Series(data=np.zeros((len(y_test))), index=y_test.index, dtype='int16')
base_accuacy = accuracy_score(y_test, base_predicts)
print(f"Accuracy of the constant model {base_accuacy:.3f}")

Accuracy of the constant model 0.900


### Logistic Regression

- First, train a logistic regression model.
- Training, hyperparameter selection and cross-validation are done with `GridSearchCV` from `sklearn`.
- The regularization hyperparameter C is tuned.

In [18]:
parameters = {'C': np.linspace(10, 20, num = 11, endpoint = True),
             'max_iter': [1000]}
lrm = LogisticRegression()
clf = GridSearchCV(lrm, parameters,
                  cv=5,
                  scoring='f1',
                  n_jobs=-1,
                  verbose=2)
clf.fit(X_train_vec, y_train)

Fitting 5 folds for each of 11 candidates, totalling 55 fits


GridSearchCV(cv=5, estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'C': array([10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.]),
                         'max_iter': [1000]},
             scoring='f1', verbose=2)

In [19]:
print(f"Best f1 in cross-validation: {clf.best_score_:.3f}")
print(f"Regularization parameter of the best model: {clf.best_params_}")

Best f1 in cross-validation: 0.773
Regularization parameter of the best model: {'C': 12.0, 'max_iter': 1000}


In [20]:
lrm = LogisticRegression(C=13, max_iter=1000)
lrm.fit(X_train_vec, y_train)
predict = lrm.predict(X_test_vec)
f1_lr = f1_score(y_test, predict)

In [21]:
print(f"f1 on the test set: {f1_lr:.3f}")

f1 on the test set: 0.775


- Run a sanity check of the model: calculate the accuracy and compare it with the constant model.

In [22]:
accuracy_lr = accuracy_score(y_test, predict)
print(f"Accuracy of the logistic regression {accuracy_lr:.3f}, higher than that of the constant model")

Accuracy of the logistic regression 0.959, higher than that of the constant model


- The f1 on the test set satisfies the requirement of the task.

### NB-SVM

- Now try the `NBSVM` model (Naive Bayes - Support Vector Machine). Here it is implemented with `LogisticRegression` with the optimization settings `solver='liblinear', dual=True`, in which case the two models behave similarly.
(idea: https://www.kaggle.com/jhoward/nb-svm-strong-linear-baseline)

In [23]:
def prob(x, y_i, y):
    p = x[y==y_i].sum(axis=0)
    return (p+1) / ((y==y_i).sum()+1)

In [24]:
r = np.log(prob(X_train_vec, 1, y_train.values) / prob(X_train_vec, 0, y_train.values))
X_train_nb = X_train_vec.multiply(r)
X_test_nb = X_test_vec.multiply(r)

In [25]:
parameters = {'C': np.linspace(1, 11, num = 11, endpoint = True)}
nblrm = LogisticRegression(solver='liblinear', 
                           dual=True, 
                           max_iter = 1000)
clf_nb = GridSearchCV(nblrm, parameters,
                  cv=5,
                  scoring='f1',
                  n_jobs=-1,
                  verbose=2)

In [26]:
clf_nb.fit(X_train_nb, y_train)

Fitting 5 folds for each of 11 candidates, totalling 55 fits


GridSearchCV(cv=5,
             estimator=LogisticRegression(dual=True, max_iter=1000,
                                          solver='liblinear'),
             n_jobs=-1,
             param_grid={'C': array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11.])},
             scoring='f1', verbose=2)

In [27]:
print(f"Best f1 in cross-validation: {clf_nb.best_score_:.3f}")
print(f"Regularization parameter of the best model: {clf_nb.best_params_}")

Best f1 in cross-validation: 0.791
Regularization parameter of the best model: {'C': 3.0}


In [28]:
nblrm = LogisticRegression(C=3,
                           solver='liblinear', 
                           dual=True,
                           max_iter=1000)
nblrm.fit(X_train_nb, y_train)
predict = nblrm.predict(X_test_nb)
f1_nblr = f1_score(y_test, predict)

In [29]:
print(f"f1 on the test set: {f1_nblr:.3f}")

f1 on the test set: 0.788


- This model differs little from the initial logistic regression (f1 grew by about 1 percentage point).
- For comparison, try the `LinearSVC` model (Linear Support Vector Classification).

### LinearSVC

In [30]:
parameters = {'C': np.linspace(1, 31, num = 7, endpoint = True)}
lsvcm = LinearSVC(max_iter = 1000)
clf_lsvc = GridSearchCV(lsvcm, parameters,
                  cv=5,
                  scoring='f1',
                  n_jobs=-1,
                  verbose=2)

In [31]:
clf_lsvc.fit(X_train_vec, y_train)

Fitting 5 folds for each of 7 candidates, totalling 35 fits


GridSearchCV(cv=5, estimator=LinearSVC(), n_jobs=-1,
             param_grid={'C': array([ 1.,  6., 11., 16., 21., 26., 31.])},
             scoring='f1', verbose=2)

In [32]:
print(f"Best f1 in cross-validation: {clf_lsvc.best_score_:.3f}")
print(f"Regularization parameter of the best model: {clf_lsvc.best_params_}")

Best f1 in cross-validation: 0.776
Regularization parameter of the best model: {'C': 1.0}


In [33]:
lsvcm = LinearSVC(C=clf_lsvc.best_params_['C'],
                  max_iter=1000)
lsvcm.fit(X_train_vec, y_train)
predict = lsvcm.predict(X_test_vec)
f1_lsvc = f1_score(y_test, predict)

In [34]:
print(f"f1 on the test set: {f1_lsvc:.3f}")

f1 on the test set: 0.779


# 3. Conclusions

- The toxicity data is loaded and processed successfully:
    - The texts are lemmatized with `WordNetLemmatizer` from `nltk`
    - Punctuation and extra spaces are removed
    - Stop words are removed (the list is taken from `nltk`)
    - The corpus is vectorized with `TfidfVectorizer`
- The models `LogisticRegression`, `NB-SVM` and `LinearSVC` are trained on the resulting data.

| Model              | f1 score |
|:-------------------|:---------|
| LogisticRegression | 0.775    |
| NB-SVM             | 0.788    |
| LinearSVC          | 0.779    |

- The quality of the models is almost the same, the difference is about 1 percentage point. The best f1 is obtained for **NB-SVM: 0.788**.
- Cross-validation and hyperparameter selection are done with GridSearchCV.
- The sanity check was done for the LogisticRegression model.